Merging and copying best practices

By implementing the following techniques, you’ll ensure data integrity and maintainability in complex pipelines.

Use | and |= for clean dictionary merges.

Remember that the last seen wins in collisions.

Distinguish between reference assignment, shallow copies, and deep copies to avoid subtle bugs.

Always prefer copy.deepcopy() when working with nested mutable structures in production code.

### Using the ** operator inside a dictionary or using the Dictionary spread operator or Dictionary Unpacking Operator

It takes all the key-value pairs from an existing dictionary and copies them directly into a new dictionary 

In Python, the ** operator inside a dictionary signifies the dictionary unpacking operator (often called the dictionary spread operator).
It takes all the key-value pairs from an existing dictionary and copies them directly into a new dictionary.

What does it do in your code?

Instead of modifying the original state dictionary in place, your functions are creating a brand-new dictionary, copying everything from the old state into it, and then adding a new key.

For example, look at the planner function:

In [ ]:

def planner(state):
    return {**state, "plan": "Retrieve policy"}


If state starts as {"query": "Sweden leave policy"}, 
the expression {**state, "plan": "Retrieve policy"} evaluates exactly like this under the hood:

    1. Unpack **state: It pulls out "query": "Sweden leave policy".
    2. Add the new key: It adds "plan": "Retrieve policy".
    3. Combine: It returns {"query": "Sweden leave policy", "plan": "Retrieve policy"}.


Now, Why use **state insetead of standard assignment ?

sing ** follows a design pattern called immutability.
1. Safety: It prevents functions from accidentally modifying or breaking the original state variable directly.
2. State History: Because a new dictionary is created at every single step, frameworks like LangGraph can easily keep track of a "history" or "checkpoint" of 

what the state looked like before and after an agent ran.

If a key already exists in state and you specify it again after the **state, the new value will overwrite the old one.
Would you like to see how to use this unpacking operator to overwrite an existing key in the state, or how it can be used to merge two entirely separate dictionaries together?



Scnario A: without ** (in-place mutation)

If you modify the state directly, your functions don't return anything (or just return the same object). You lose all track of what happened in the past.


In [8]:
def planner(state):
    state["plan"] = "Retrieve Policy"
    return state

original_state = {"query": "Sweden Leave Policy"}

print ("Without using **")
print (f"original_state (before calling the planner1 method): {original_state}")
new_state = planner(original_state)
print (f"original_state (after calling the planner1 method): {original_state}")
print (f"new_state (after calling the planner2 method): {new_state}")
print (original_state is new_state)

print () 
print ()
print ("With using **")
# Now use dictionary unpacking with ** 
prev_state = {"query": "Sweden Leave Policy"}
def planner2(state):
    return {**state, "plan": "Sweden Leave Policy"}

print (f"prev_state (before calling the planner1 method): {prev_state}")
new_state = planner2(prev_state)
print (f"prev_state (after calling the planner1 method): {prev_state}")
print (f"new_state (after calling the planner2 method): {new_state}")
print (original_state is new_state)


Without using **
original_state (before calling the planner1 method): {'query': 'Sweden Leave Policy'}
original_state (after calling the planner1 method): {'query': 'Sweden Leave Policy', 'plan': 'Retrieve Policy'}
new_state (after calling the planner2 method): {'query': 'Sweden Leave Policy', 'plan': 'Retrieve Policy'}
True


With using **
prev_state (before calling the planner1 method): {'query': 'Sweden Leave Policy'}
prev_state (after calling the planner1 method): {'query': 'Sweden Leave Policy'}
new_state (after calling the planner2 method): {'query': 'Sweden Leave Policy', 'plan': 'Sweden Leave Policy'}
False



Real-World Benefit for Agent Workflows

If your workflow hits an error at the summarizer step, look at how debugging changes:

• Without **: Your state variable is a messy, half-broken mix of data. You cannot easily see what the state looked like right before the crash because the data was overwritten in place.

• With **: You can track the state history at every step (state_1, state_2, state_3). If step 3 fails, you can easily inspect state_2 to see exactly what bad data the researcher handed to the summarizer, or even rerun step 3 with fixed code without restarting the whole pipeline.

